# t-SNE para reducción de dimensionalidad
**Autor:** Autoagente de Notebooks  
**Fecha:** 2026-06-11  
**Tags:** reducción de dimensionalidad, t-SNE, visualización, aprendizaje no supervisado  
**Propósito:** Presentar t-SNE de forma pedagógica y reproducible, con un ejemplo sintético de visualización de clases.

## Descripción
El algoritmo t-SNE calcula una medida de similitud entre pares de instancias en el espacio de alta dimensión y en el espacio de baja dimensión. Esto se logra estableciendo las probabilidades del espacio de baja dimensión para que sean similares a las del espacio de alta dimensión. Medimos la diferencia entre las distribuciones de probabilidad de ambos espacios utilizando la divergencia de Kullback-Leibler e intentamos optimizarla.

## Objetivo del modelo
t-SNE (t-distributed Stochastic Neighbor Embedding) es una técnica de reducción de dimensionalidad especialmente útil para visualización de datos de alta dimensión en 2 o 3 dimensiones.
Su objetivo es preservar la estructura local de los datos y revelar agrupamientos o patrones que no son fáciles de ver en el espacio original.

## Estructura del modelo
- Similaridad en el espacio original entre dos puntos $x_i$ y $x_j$.
- Distribución de probabilidad condicional $p_{j|i}$ basada en distancias gaussianas.
- Distribución de probabilidad conjunta $q_{ij}$ en el espacio reducido con Student t de un grado de libertad.
- Minimización de divergencia de Kullback-Leibler entre las distribuciones $P$ y $Q$.

## Modelo matemático
Definimos primero las probabilidades en el espacio original: con un parámetro de perplexity $\sigma_i$, la probabilidad condicional de que $x_i$ elija a $x_j$ como vecino es:

$$
p_{j|i} = \frac{\exp\left(-\|x_i - x_j\|^2 / 2\sigma_i^2\right)}{\sum_{k \neq i} \exp\left(-\|x_i - x_k\|^2 / 2\sigma_i^2\right)}.
$$ 

A continuación, obtenemos la probabilidad simétrica promedio:

$$
p_{ij} = \frac{p_{j|i} + p_{i|j}}{2n}.
$$ 

En el espacio reducido de coordenadas $y_i$, la probabilidad conjunta se define con una distribución de Student $t$ con un grado de libertad:

$$
q_{ij} = \frac{\left(1 + \|y_i - y_j\|^2\right)^{-1}}{\sum_{k \neq l} \left(1 + \|y_k - y_l\|^2\right)^{-1}}.
$$ 

La función de pérdida minimizada es la divergencia de Kullback-Leibler entre las distribuciones $P$ y $Q$:

$$
C = \sum_{i \neq j} p_{ij} \log \frac{p_{ij}}{q_{ij}}.
$$ 

Este enfoque preserva la estructura local de los datos y evita el problema de "crowding" que tienen otros métodos simples.


## Ventajas y desventajas
- Ventaja: revela agrupamientos locales y relaciones no lineales en datos de alta dimensión.
- Ventaja: es útil para análisis exploratorio y detección de estructuras en clases.
- Desventaja: es computacionalmente costoso para muchos datos y no es un método paramétrico.
- Desventaja: los resultados pueden variar según la `perplexity` y la inicialización.
- Nota: t-SNE no es ideal como paso de preprocesamiento para modelos que requieren transformar nuevos datos sin recalcular el embedding.

## Aplicaciones
- Visualización de datos en clasificación de imágenes y texto.
- Exploración de clusters en datos de genómica o bioinformática.
- Detección de anomalías y agrupamiento en datos de comportamiento.
- Inspección de representaciones internas en redes neuronales profundas.

In [ ]:
# Imports y configuración
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

In [ ]:
# Crear datos sintéticos
X, y = make_classification(
    n_samples=500,
    n_features=20,
    n_informative=10,
    n_redundant=5,
    n_clusters_per_class=1,
    n_classes=3,
    random_state=RANDOM_SEED
)

X = pd.DataFrame(X, columns=[f'x{i}' for i in range(X.shape[1])])
y = pd.Series(y, name='target')

print('Dimensiones:', X.shape)
print('Clases:', y.value_counts().sort_index())
X.head()

In [ ]:
# Separar train/test y preparar pipeline simple
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_SEED
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print('Train:', X_train_s.shape, 'Test:', X_test_s.shape)

In [ ]:
# Aplicar t-SNE al conjunto completo escalado para visualización
X_s = scaler.transform(X)
tsne = TSNE(n_components=2, perplexity=30, random_state=RANDOM_SEED, init='pca')
X_embedded = tsne.fit_transform(X_s)

df_embedded = pd.DataFrame(X_embedded, columns=['dim1', 'dim2'])
df_embedded['target'] = y.values

plt.figure(figsize=(8, 6))
for label in sorted(df_embedded['target'].unique()):
    mask = df_embedded['target'] == label
    plt.scatter(df_embedded.loc[mask, 'dim1'], df_embedded.loc[mask, 'dim2'], alpha=0.7, label=f'Clase {label}')
plt.title('t-SNE sobre datos sintéticos')
plt.xlabel('Dimensión 1')
plt.ylabel('Dimensión 2')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Evaluación de la separación de clusters
score = silhouette_score(X_embedded, y)
print(f'Silhouette score en el espacio t-SNE: {score:.3f}')

print('Nota: esta métrica se calcula sobre el embedding y ayuda a cuantificar la separación relativa de clases en el espacio reducido.')

## Conclusiones y siguientes pasos
- t-SNE es muy útil para visualizar datos de alta dimensión cuando se busca explorar la separación entre grupos.
- En este ejemplo, el embedding mostró cómo las clases sintéticas se organizan en el plano reducido.
- Es importante recordar que t-SNE no es un modelo paramétrico y que los resultados dependen de la `perplexity` y la inicialización.
- Para producción o sistemas que necesiten transformar datos nuevos, conviene considerar alternativas paramétricas o métodos como PCA/UMAP.

## Reproducibilidad y dependencias
Instala las dependencias con:
```bash
pip install numpy pandas scikit-learn matplotlib jupyter
```

Asegúrate de ejecutar el notebook en un entorno con Python 3.8+ y de fijar `RANDOM_SEED = 42` para obtener resultados consistentes.

### Referencias
- t-SNE en scikit-learn: https://scikit-learn.org/stable/modules/generated/sklearn.manifold.TSNE.html
- Artículo original de van der Maaten y Hinton sobre t-SNE.